In [14]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
import os
from pathlib import Path
from datetime import datetime
from IPython.display import Markdown, display

load_dotenv()

True

## Part 1: Long Term Memory

In [15]:
memory_path = os.path.abspath("memory/memory.json")
memory_params = {"command": "npx",
                 "args": ["-y", "@modelcontextprotocol/server-memory"],
                 "env": {"MEMORY_FILE_PATH": memory_path}}

async with MCPServerStdio(
            params=memory_params,
            client_session_timeout_seconds=60) as server:
    memory_tools = await server.list_tools()

memory_tools

[Tool(name='create_entities', title='Create Entities', description='Create multiple new entities in the knowledge graph. An entity whose name already exists, or repeats an earlier entity in the same call, is skipped and its observations are not added; the result lists the skipped names in `skipped`. Use add_observations to add observations to an existing entity.', inputSchema={'type': 'object', '$schema': 'https://json-schema.org/draft/2020-12/schema', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string', 'description': 'The name of the entity'}, 'entityType': {'type': 'string', 'description': 'The type of the entity'}, 'observations': {'type': 'array', 'items': {'type': 'string'}, 'description': 'An array of observation contents associated with the entity'}}, 'required': ['name', 'entityType', 'observations']}}}, 'required': ['entities']}, outputSchema={'$schema': 'https://json-schema.org/draft/2020-12/schema', 'type': 'obje

In [16]:
instructions = "You use your entity tools as a persistent memory to store and recall information about your conversations."
request = """My name is Michael. I'm an LLM engineer. I'm taking a course on AI agents, including the incredible MCP protocol. MCP is a protocol for 
connecting agents with tools, resources, and prompt templates, and makes it easy to integrate AI agents with capabilities."""
model = "gpt-4o-mini"

In [17]:
async with MCPServerStdio(
    params=memory_params,
    client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(
        name="agent",
        instructions=instructions,
        model=model,
        mcp_servers=[mcp_server]
    )
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


I've noted that your name is Michael and that you're an LLM engineer currently taking a course on AI agents, including the MCP protocol. Your observations about the MCP protocol have been added. If there's anything specific you'd like to discuss or any questions you have, feel free to share!

In [18]:
async with MCPServerStdio(
        params=memory_params,
        client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(
        name="agent",
        instructions=instructions,
        model=model,
        mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


I've noted your information about being an LLM engineer and taking a course on AI agents, including the MCP protocol. However, it seems that the details about you and the MCP protocol were already recorded in the system, so no new observations were added.

If there's anything specific you'd like to update or any additional information you'd like to share, feel free to let me know!

## Part 2: Web Search

In [19]:
tavily_params = {"command": "npx", 
    "args": ["-y", "tavily-mcp@latest"],
    "env": {"TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")}}
async with MCPServerStdio(
    params=tavily_params,
    client_session_timeout_seconds=60) as server:
    tavily_tools = await server.list_tools()

tavily_tools


[Tool(name='tavily_search', title=None, description='Search the web for current information on any topic. Use for news, facts, or data beyond your knowledge cutoff. Returns snippets and source URLs.', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query'}, 'search_depth': {'type': 'string', 'enum': ['basic', 'advanced', 'fast', 'ultra-fast'], 'description': "The depth of the search. 'basic' for generic results, 'advanced' for more thorough search, 'fast' for optimized low latency with high relevance, 'ultra-fast' for prioritizing latency above all else", 'default': 'basic'}, 'topic': {'type': 'string', 'enum': ['general'], 'description': 'The category of the search. This will determine which of our agents will be used for the search', 'default': 'general'}, 'time_range': {'type': 'string', 'description': 'The time range back from the current date to include in the search results', 'enum': ['day', 'week', 'month', 'year']}, 'start_date':

In [20]:
instructions = "You search the web for information and briefly summarize the takeaways."
request = f"""Please research the latest news on Amazon stock price and briefly summarize its outlook.
For context, the current date is {datetime.now().strftime('%Y-%m-%d')}
"""
model = "gpt-4o-mini"
search_only = create_static_tool_filter(allowed_tool_names=["tavily_search"])


In [21]:
async with MCPServerStdio(
        params=tavily_params,
        client_session_timeout_seconds=60,
        tool_filter=search_only) as mcp_server:
    agent = Agent(
        name="agent",
        instructions=instructions,
        model="gpt-4o-mini",
        mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))



As of November 2, 2026, Amazon's stock (AMZN) is trading around **$241.46**, reflecting a slight decrease from previous highs. The stock has shown some short-term volatility but has maintained a **7.81% year-to-date return** and a **1.90% return over the past year**.

### Key Factors Influencing Outlook:

1. **Strong AWS Growth**: The company's growth prospects primarily hinge on the performance of Amazon Web Services (AWS), which is seeing increased demand due to the rise of AI technologies. Analysts are closely watching AWS revenue and operating margins.

2. **Upcoming Earnings Reports**: The release of Q3 earnings is a critical upcoming event. Investors are eager for indicators of sustained growth amid heavy capital expenditures on data centers.

3. **Retail Performance and Seasonality**: Seasonal factors like holiday shopping trends will also play a significant role in shaping revenue in the upcoming months.

4. **Regulatory Scrutiny**: Ongoing concerns regarding regulatory actions in both the U.S. and European Union could impact stock performance if they impose restrictions on the company's operations.

5. **Market Competition**: Competition from other tech giants, particularly Microsoft Azure and Google Cloud, is a significant factor that may pressure Amazon's growth and profitability.

### Predictions:
- Analysts predict short-term fluctuations with possible prices ranging between **$255 and $277** by the end of November.
- Longer-term forecasting indicates a potential stock price rise to approximately **$300** to **$334.90** by the end of 2027, given sustained growth in key service areas.

Overall, while short-term uncertainties exist, the consensus among analysts suggests optimism regarding Amazon's growth trajectory, contingent on AWS performance and upcoming retail forecasts.

## Part 3: Agentic RAG

In [22]:
vectordb_path = Path("memory/qdrant")
vectorstore_params = {
    "command": "uvx",
    "args": ["mcp-server-qdrant"],
    "env": {
        "QDRANT_LOCAL_PATH": str(vectordb_path),
        "COLLECTION_NAME": "knowledge",
    },
}

async with MCPServerStdio(
    params=vectorstore_params,
    client_session_timeout_seconds=120
) as server:
    vectorstore_tools = await server.list_tools()
vectorstore_tools

[Tool(name='qdrant-find', title=None, description='Look up memories in Qdrant. Use this tool when you need to: \n - Find memories by their content \n - Access memories for further analysis \n - Get some personal information about the user', inputSchema={'properties': {'query': {'description': 'What to search for', 'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='qdrant-store', title=None, description='Keep the memory for later use, when you are asked to remember something.', inputSchema={'properties': {'information': {'description': 'Text to store', 'title': 'Information', 'type': 'string'}, 'metadata': {'anyOf': [{'additionalProperties': True, 'type': 'object'}, {'type': 'null'}], 'default': None, 'description': 'Extra metadata stored along with memorised information. Any json is accepted.', 'title': 'Metadata'}}, 'required': ['information'], 'type': 'object'}, outpu

In [23]:
INSTRUCTIONS = """ You research topics on the web and build up a knowledge base for later.
When you learn something worth keeping, store it in your knowledge base. When you are asked
what you know, search your knowledge base and answer from it."""

model = "gpt-4o-mini"

async with MCPServerStdio(
    params=tavily_params,
    client_session_timeout_seconds=60,
    tool_filter = search_only
) as search_server:
    async with MCPServerStdio(
        params=vectorstore_params,
        client_session_timeout_seconds=120) as vector_server:
            agent = Agent(
                name="researcher",
                instructions=INSTRUCTIONS,
                model=model,
                mcp_servers=[search_server, vector_server])
            with trace("research and store"):
                result = await Runner.run(agent, " Research the latest news on Nvidia and store the key facts in your knowledge base.", max_turns=20)
            display(Markdown(result.final_output))


I've stored the latest key facts about Nvidia in my knowledge base:

1. **China's Anti-Monopoly Concerns**: China has accused Nvidia of breaching anti-monopoly laws, potentially disrupting its operations and affecting enterprises reliant on its GPUs, particularly amidst rising US-China trade tensions (September 16, 2025).

2. **New AI GPUs Released**: Nvidia launched a new purpose-built GPU for AI inferencing aimed at large workloads, accompanied by a next-generation platform for complex processing and generative video (September 9, 2025).

3. **Restarting H20 AI Chip Sales**: Plans to restart H20 AI chip sales to China have been announced, with compliance to new export rules and expected approval from the US government (July 15, 2025).

4. **Investment in CoreWeave**: Nvidia invested $2 billion in CoreWeave, a cloud service provider built on its infrastructure, reflecting confidence in CoreWeave's growth strategy (February 2, 2026).

5. **Security Alert**: Nvidia issued a warning regarding potential vulnerabilities of its GPUs to Rowhammer attacks, advising developers and manufacturers to be cautious (July 15, 2025).

Feel free to ask if you need more information!

In [24]:
async with MCPServerStdio(
    params=vectorstore_params,
    client_session_timeout_seconds=120) as vector_server:
    agent = Agent(
        name="researcher",
        instructions=INSTRUCTIONS,
        model=model,
        mcp_servers=[vector_server])
    with trace("retrieve"):
        result = await Runner.run(agent, "Based on your knowledge base, what's the latest news on Nvidia?")
    display(Markdown(result.final_output))

Here are the latest updates on Nvidia:

1. **China's Anti-Monopoly Concerns**: China has accused Nvidia of breaching anti-monopoly laws, which could disrupt its global operations amid ongoing US-China trade tensions. (September 16, 2025)

2. **New AI GPUs Released**: Nvidia has launched new GPUs designed for AI inferencing, aimed at handling large workloads, along with a next-generation platform for generative video processing. (September 9, 2025)

3. **H20 AI Chip Sales Resumption**: The company plans to resume sales of H20 AI chips to China, having applied for US government approval. Indications suggest that licenses may be granted soon. (July 15, 2025)

4. **New Professional GPUs**: Nvidia introduced the RTX Pro 4000 SFF and RTX Pro 2000, designed for compact AI workstations optimized for efficiency. (August 12, 2025)

5. **GenAI Model for Robotics**: A new generative AI model named Cosmos Reason helps robots analyze video and graphics data to make human-like decisions.

6. **Security Alerts**: Nvidia has issued warnings regarding potential vulnerabilities in its GPUs to Rowhammer attacks, emphasizing the need for caution among developers. (July 15, 2025)

7. **Expanding Manufacturing Capacity**: The company is making efforts to expand global manufacturing capacity to meet the increasing demand for AI infrastructure.

8. **New Platform for AI Factories**: Nvidia has developed a platform named Minerva for end-to-end observability in AI factories.

These developments indicate Nvidia is actively expanding its capabilities while navigating complex global challenges.

# Part 4: Integrations

In [27]:
massive_api_key = os.getenv("MASSIVE_API_KEY")

if massive_api_key:
    market_params = {
        "command": "uvx",
        # mcp_massive still imports mcp.server.fastmcp, an API the mcp SDK's 2.0.0 release removed.
        # mcp_massive doesn't cap its mcp dependency, so pin uvx to an older, compatible mcp here.
        "args": ["--with", 
        "mcp<2.0.0", 
        "--from",
        "git+https://github.com/massive-com/mcp_massive@v0.10.0", "mcp_massive"
        ],
        "env": {"MASSIVE_API_KEY": massive_api_key}
    }
else:
    market_params = {
        "command": "uv",
        "args": ["run", 
                 "-m",
                 "backend.market_server"]}
async with MCPServerStdio(
    params=market_params,
    client_session_timeout_seconds=120
) as server:
    market_tools = await server.list_tools()

market_tools

[Tool(name='search_endpoints', title=None, description='Search for market data API endpoints and built-in finance functions by natural language query. Use this FIRST to find the right endpoint before calling call_api. Covers stocks, options, forex, crypto, futures, indices, ETFs, and economic data. Pass market to pin results to a specific asset class when you already know it; omit it and the server will infer from the query. Use detail="more" to see query parameter docs needed for building call_api requests.', inputSchema={'additionalProperties': False, 'properties': {'query': {'description': 'Natural language search query for API endpoints', 'minLength': 1, 'title': 'Query', 'type': 'string'}, 'scope': {'anyOf': [{'enum': ['all', 'endpoints', 'functions'], 'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Search scope: "endpoints" for API only, "functions" for local functions only, or "all"/omit for both', 'title': 'Scope'}, 'max_results': {'anyOf': [{'maximum': 2

In [34]:
from datetime import datetime
from zoneinfo import ZoneInfo

from agents import Agent, ModelSettings, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
from pydantic import BaseModel


class ClosePrice(BaseModel):
    ticker: str
    close_price: float
    close_date: str  # YYYY-MM-DD
    currency: str


now_et = datetime.now(ZoneInfo("America/New_York")).strftime("%Y-%m-%d %H:%M %Z")

instructions = f"""
You answer questions about the stock market.
The current time is {now_et}.

Always get market data from the tools, and never answer from memory.
Report the most recent completed trading session's close.
Don't use a partial price from a session that's still open.
If the tool fails or the data is missing, say so instead of estimating.
"""

request = "What was the last close price for Apple (AAPL)?"
model = "gpt-5.4-mini"

async with MCPServerStdio(
    params=market_params,
    client_session_timeout_seconds=120,
    cache_tools_list=True,
    tool_filter=create_static_tool_filter(
        # Massive's MCP server tools, plus the local fallback server's tool
        allowed_tool_names=["search_endpoints", "call_api", "query_data", "lookup_share_price"]
    ),
) as mcp_server:
    agent = Agent(
        name="market_agent",
        instructions=instructions,
        model=model,
        mcp_servers=[mcp_server],
        model_settings=ModelSettings(tool_choice="required"),
        output_type=ClosePrice,
    )

    with trace("last_close_lookup"):
        result = await Runner.run(agent, request, max_turns=5)

quote = result.final_output
display(Markdown(
    f"**{quote.ticker}** closed at **{quote.close_price:.2f} {quote.currency}** "
    f"on {quote.close_date}."
))

**AAPL** closed at **344.63 USD** on 2026-10-30.

In [32]:
instructions = "You answer questions about the stock market."
request = "What was the the last close price that Apple (AAPL) traded at?"
model ="gpt-5.4-mini"

async with MCPServerStdio(
    params=market_params,
    client_session_timeout_seconds=120
) as mcp_server:
    agent = Agent(
        name="agent",
        instructions=instructions,
        model=model,
        mcp_servers=[mcp_server]
    )
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

I’m unable to fetch AAPL’s live last close price from this data source because the market data endpoint isn’t authorized on this account.

If you want, I can still help in one of these ways:
- show you how to get the latest close from a public source,
- help you look up AAPL’s recent historical closes if you have another data provider,
- or explain how to calculate the “last close” from end-of-day data.